In [1]:
%pip install keras matplotlib numpy pandas scikit-learn tensorflow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
from tensorflow import keras
from keras.datasets import fashion_mnist

#loading the dataset
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

#normalizing the data
X_train_norm = X_train.reshape((X_train.shape[0], 28, 28, 1)).astype('float32') / 255.0
X_test_norm = X_test.reshape((X_test.shape[0], 28, 28, 1)).astype('float32') / 255.0

print(X_train_norm.shape)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 3us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
(60000, 28, 28, 1)


In [8]:
#one hot encoding the labels
def one_hot_encodation(y, num_classes=10):
    one_hot = np.zeros((y.size, num_classes))
    one_hot[np.arange(y.size), y] = 1
    return one_hot

y_train_encoded = one_hot_encodation(y_train)
y_test_encoded = one_hot_encodation(y_test)

print(y_train_encoded[0])


#explaining the function
# For each label in y, we create a zero vector of length num_classes (10 for Fashion-MNIST).
# We then set the index corresponding to the label to 1.

#then why in output we see 0s and 1s only in the array?

#  because we are creating a one-hot encoded representation of the labels. and 0's means absence of that class and 1 means presence of that class.

#as we see below in the output its a seqnuence of 0s at first which means absence of that class and 1 at index 9 which means presence of that class



[0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]


In [31]:
#relu activation function
def relu(x):
    return np.maximum(0, x)

#derivative of relu
def relu_derivative(x):
    return (x>0).astype(float)

#softwax function
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

#cross entropy loss function
def cross_entropy_loss(y_true, y_pred):
    m = y_true.shape[0]

    log_likelihood = -np.sum(y_true * np.log(y_pred + 1e-9), axis=1)
    loss = np.sum(log_likelihood) / m
    return loss



In [ ]:
#weight initialisating function
def initialisation_weights(input_dimn, hidden1,hidden2, hidden3, output_dimn):
   
   params = {}
   params['W1'] = np.random.randn(input_dimn, hidden1) * np.sqrt(2.0/input_dimn)
   params['b1'] = np.zeros((1, hidden1))

   params['W2'] = np.random.randn(hidden1, hidden2) * np.sqrt(2.0/hidden1)
   params['b2'] = np.zeros((1, hidden2))

   params['W3'] = np.random.randn(hidden2, hidden3) * np.sqrt(2.0/hidden2)
   params['b3'] = np.zeros((1, hidden3))

   params['W4'] = np.random.randn(hidden3, output_dimn) * np.sqrt(2.0/hidden3)
   params['b4'] = np.zeros((1, output_dimn))

   return params


In [33]:
#explanation of weight initialisation function

# This function initializes the weights and biases for a neural network with three hidden layers.

# It takes the input dimension, sizes of the three hidden layers, and output dimension as arguments.

# It returns a dictionary containing the weights and biases for each layer, initialized using a random normal distribution scaled by the square root of the number of input units to that layer (He initialization).

# The biases are initialized to zero.

# This initialization helps in maintaining the variance of activations through the layers, which can lead to better convergence during training.



In [34]:
#forward pass function

def forward_pass(X, params):
    cache = {}

    #layer 1:
    Z1 = np.dot(X, params['W1']) + params['b1']
    A1 = relu(Z1)
    cache['Z1'] = Z1
    cache['A1'] = A1

    #layer 2:
    Z2 = np.dot(A1, params['W2']) + params['b2']
    A2 = relu(Z2)
    cache['Z2'] = Z2
    cache['A2'] = A2

    #layer 3:
    Z3 = np.dot(A2, params['W3']) + params['b3']
    A3 = relu(Z3)
    cache['Z3'] = Z3
    cache['A3'] = A3

    #OUTPUT LAYER:
    Z4 = np.dot(A3, params['W4']) + params['b4']
    A4 = softmax(Z4)
    cache['Z4'] = Z4
    cache['A4'] = A4

    return A4, cache


In [35]:
#explanation of forward pass function

# This function performs the forward pass through a neural network with three hidden layers and an output layer.

# It takes the input data X and a dictionary of parameters (weights and biases) as arguments.

# It computes the linear combinations (Z) and activations (A) for each layer using the ReLU activation function for the hidden layers and softmax for the output layer.

# It stores the intermediate values (Z and A) in a cache dictionary for use in backpropagation.

# It returns the output activations (A4) and the cache dictionary.



In [44]:
#backward pass function:


def backward_function(X, y_true, params, cache):


    grads = {}


    m = X.shape[0]


    dz4 = cache['A4'] - y_true
    grads['dw4'] = np.dot(cache['A3'].T, dz4) / m
    grads['db4'] = np.sum(dz4, axis=0, keepdims=True) / m   


    #layer 3
    dA3 = np.dot(dz4, params['W4'].T)
    dZ3 = dA3 * relu_derivative(cache['Z3'])
    grads['dw3'] = np.dot(cache['A2'].T, dZ3) / m
    grads['db3'] = np.sum(dZ3, axis=0, keepdims=True) / m


    #layer 2
    dA2 = np.dot(dZ3, params['W3'].T)
    dZ2 = dA2 * relu_derivative(cache['Z2'])
    grads['dw2'] = np.dot(cache['A1'].T, dZ2) / m
    grads['db2'] = np.sum(dZ2, axis=0, keepdims=True) / m


    #layer 1
    dA1 = np.dot(dZ2, params['W2'].T)
    dZ1 = dA1 * relu_derivative(cache['Z1'])
    grads['dw1'] = np.dot(X.T, dZ1) / m
    grads['db1'] = np.sum(dZ1, axis=0, keepdims=True) / m


    return grads



In [37]:
#explanation of backward pass function

# This function performs the backward pass (backpropagation) through a neural network with three hidden layers and an output layer.

# It takes the input data X, true labels y_true, a dictionary of parameters (weights and biases), and a cache dictionary containing intermediate values from the forward pass as arguments.

# It computes the gradients of the loss with respect to the weights and biases for each layer using the chain rule of calculus.

# It returns a dictionary containing the gradients for each weight and bias.



In [38]:
#weight update function
def update_weights(params, grads, learn_rate):
    params['W1'] -= learn_rate * grads['dw1']
    params['b1'] -= learn_rate * grads['db1']

    params['W2'] -= learn_rate * grads['dw2']
    params['b2'] -= learn_rate * grads['db2']

    params['W3'] -= learn_rate * grads['dw3']
    params['b3'] -= learn_rate * grads['db3']

    params['W4'] -= learn_rate * grads['dw4']
    params['b4'] -= learn_rate * grads['db4']

    return params


In [39]:
#explanation of the weight update function

# This function updates the weights and biases of a neural network using gradient descent.

# It takes a dictionary of parameters (weights and biases), a dictionary of gradients, and a learning rate as arguments.

# It updates each weight and bias by subtracting the product of the learning rate and the corresponding gradient.

# It returns the updated parameters dictionary.



In [46]:
#defining the predict function

def predict(X, params):
    A4, _ = forward_pass(X, params)
    predictions = np.argmax(A4, axis=1)
    return predictions


In [25]:
#explaining the predict function

# This function makes predictions using the trained neural network.
# It takes the input data X and a dictionary of parameters (weights and biases) as arguments.
# It performs a forward pass through the network to compute the output activations.
# It returns the predicted class labels by selecting the index of the maximum activation for each input sample.


In [ ]:

params = initialisation_weights(784, 128, 64, 32, 10)


#training parameters
epoch = 20
learning_rate = 0.01
batch_size = 64


#training loop
for e in range(epoch):

    #np.random.permutation shuffles the data at each epoch
    permutation = np.random.permutation(X_train_norm.shape[0])

    #X_shuffled and y_shuffled are the shuffled versions of X_train_norm and y_train_encoded respectively

    X_shuffled = X_train_norm[permutation]
    y_shuffled = y_train_encoded[permutation]

    #mini-batch gradient descent on the shuffled data

    #range is from 0 to number of samples in X_train_norm with step size of batch_size

    #so here X_train_norm.shape[0] is 60000 as there are 60000 samples in training data

    #so range(0, 60000, 64) will generate numbers from 0 to 60000 with step size of 64. here it will be a batch of 64 samples at a time

    for i in range(0, X_train_norm.shape[0], batch_size):

        # Extract the current mini-batch of data
        #i: starting index of the batch
        #i+batch_size: ending index of the batch

        X_batch = X_shuffled[i:i+batch_size]
        y_batch = y_shuffled[i:i+batch_size]

        #reshaping the X_batch to (batch_size, 28*28) from (batch_size, 28, 28, 1)
        #because our model input layer is of size 28*28
        #caching the forward pass output
        #so we need to reshape it

        X_batch = X_batch.reshape(X_batch.shape[0], 28*28)

        A4, cache = forward_pass(X_batch, params)

        #calculating the loss by cross entropy loss function

        loss = cross_entropy_loss(y_batch, A4)


        #calculating the gradients by backward function

        grads = backward_function(X_batch, y_batch, params, cache)

        #calculating the accuracy
        #here np.argmax(y_batch, axis=1) gives the true labels from one hot encoded labels
        #so if predictions match with true labels then its correct prediction

        #thus, we calculate the mean of correct predictions to get accuracy
        

        predictions = predict(X_batch, params)
        accuracy = np.mean(predictions == np.argmax(y_batch, axis=1))
        params = update_weights(params, grads, learning_rate)

    print(f"Epoch {e+1}/{epoch}, Loss: {loss:.4f}, Accuracy: {accuracy:.4f}")
    print("---------------------------------------------------")
    accuracy_in_percentage = accuracy * 100
    print(f"Accuracy in percentage: {accuracy_in_percentage:.2f}%")


Epoch 1/20, Loss: 2.3029, Accuracy: 0.2188
---------------------------------------------------
Accuracy in percentage: 21.88%
Epoch 2/20, Loss: 2.3022, Accuracy: 0.0625
---------------------------------------------------
Accuracy in percentage: 6.25%
Epoch 3/20, Loss: 2.3023, Accuracy: 0.0312
---------------------------------------------------
Accuracy in percentage: 3.12%
Epoch 4/20, Loss: 2.3029, Accuracy: 0.0312
---------------------------------------------------
Accuracy in percentage: 3.12%
Epoch 5/20, Loss: 2.3028, Accuracy: 0.0938
---------------------------------------------------
Accuracy in percentage: 9.38%
Epoch 6/20, Loss: 2.3022, Accuracy: 0.1250
---------------------------------------------------
Accuracy in percentage: 12.50%
Epoch 7/20, Loss: 2.3023, Accuracy: 0.1562
---------------------------------------------------
Accuracy in percentage: 15.62%
Epoch 8/20, Loss: 2.3027, Accuracy: 0.0312
---------------------------------------------------
Accuracy in percentage: 3.1